<a href="https://colab.research.google.com/github/yuri-maradini/TempSal/blob/main/src/train_ueyes_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning TempSAL su UEyes — Colab

Notebook pronto per lanciare il training vero (Step 4) su GPU, invece che sulla CPU locale.

**Prima di eseguire questo notebook**, su Google Drive crea una cartella (default atteso: `MyDrive/TempSAL_UEyes/`) contenente:
- `multilevel_tempsal.pt` — il checkpoint pre-addestrato originale
- `data_ueyes.zip` — l'archivio di `data_ueyes/` generato in locale (Step 1-2)

Poi: **Runtime → Cambia tipo di runtime → GPU**, prima di eseguire le celle.

In [1]:
# Controllo che sia stata assegnata una GPU
!nvidia-smi

Tue Sep  1 11:31:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# Cambia questo path se hai usato un nome/percorso diverso su Drive
DRIVE_DIR = '/content/drive/MyDrive/TempSAL_UEyes'

assert os.path.isdir(DRIVE_DIR), (
    f"Cartella non trovata: {DRIVE_DIR}\n"
    "Creala su Drive e caricaci multilevel_tempsal.pt + data_ueyes.zip prima di continuare."
)
print('Contenuto trovato su Drive:', os.listdir(DRIVE_DIR))

Contenuto trovato su Drive: ['multilevel_tempsal.pt', 'data_ueyes.zip']


In [6]:
# Codice: sempre aggiornato da GitHub, non serve preparazione
!git clone https://github.com/yuri-maradini/TempSal.git /content/TempSal

Cloning into '/content/TempSal'...
remote: Enumerating objects: 188, done.
remote: Counting objects: 100% (188/188), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 188 (delta 45), reused 179 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (188/188), 10.26 MiB | 14.92 MiB/s, done.
Resolving deltas: 100% (45/45), done.


In [7]:
import shutil

os.makedirs('/content/TempSal/src/checkpoints', exist_ok=True)
shutil.copy(
    f'{DRIVE_DIR}/multilevel_tempsal.pt',
    '/content/TempSal/src/checkpoints/multilevel_tempsal.pt',
)
print('Checkpoint copiato.')

Checkpoint copiato.


In [8]:
import time
import zipfile

# Estratto sul disco locale di Colab (veloce), non lasciato sul mount di Drive
# (l'I/O su Drive montato e' molto piu' lento per tanti file piccoli, e qui
# ce ne sono migliaia tra immagini, mappe e volumi temporali).
t0 = time.time()
with zipfile.ZipFile(f'{DRIVE_DIR}/data_ueyes.zip') as zf:
    zf.extractall('/content/TempSal/')
print(f'Dati estratti in {time.time() - t0:.0f}s')

Dati estratti in 38s


In [9]:
# Controllo veloce di integrita': i conteggi devono combaciare con quelli
# verificati in locale (1872 train / 108 val per ciascuna sottocartella)
for sub in ['images', 'maps', 'fixation_maps', 'saliency_volumes_5', 'fixation_volumes_5']:
    for split in ['train', 'val']:
        d = f'/content/TempSal/data_ueyes/{sub}/{split}'
        n = len(os.listdir(d)) if os.path.isdir(d) else 'MANCANTE'
        print(f'{sub:22s} {split:5s} -> {n}')

images                 train -> 1872
images                 val   -> 108
maps                   train -> 1872
maps                   val   -> 108
fixation_maps          train -> 1872
fixation_maps          val   -> 108
saliency_volumes_5     train -> 9360
saliency_volumes_5     val   -> 540
fixation_volumes_5     train -> 9360
fixation_volumes_5     val   -> 540


In [10]:
# Colab ha gia' PyTorch con supporto CUDA preinstallato: installiamo solo le
# altre dipendenze del progetto, senza toccare torch/torchvision/torchaudio
# (forzare i pin usati in locale, pensati per una build CPU, rischierebbe di
# rimpiazzare la build CUDA gia' pronta di Colab con una incompatibile).
!pip install -q wandb pycocotools ftfy einops clip-anytorch kornia regex

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 120.0 MB/s eta 0:00:00


In [11]:
# Verifica che l'installazione sopra non abbia rovinato il supporto CUDA di torch
import torch
print('torch', torch.__version__, '| CUDA disponibile:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU non disponibile: controlla Runtime > Cambia tipo di runtime > GPU'

torch 2.11.0+cu128 | CUDA disponibile: True


## wandb (consigliato per questa run)

Il primo run (10 epoche) è stato fatto con `WANDB_MODE=disabled`: nessuna curva salvata, i numeri per epoca sono stati recuperati a mano dall'output della cella di training. Per questa run vale la pena accendere il logging vero, così le curve di CC/KLDIV/NSS/SIM (aggregate e per-slice) restano disponibili per il confronto e per la tesi senza dover rileggere l'output della cella.

Esegui la cella sotto (chiede l'API key, la trovi su wandb.ai/authorize) prima di lanciare il training. Se preferisci comunque saltarlo, aggiungi di nuovo `WANDB_MODE=disabled` (o `=offline` per salvare i log in locale senza account) davanti al comando `python train.py` nella cella di training.</cell id="HNcrR_LSgcXM">


In [ ]:
import wandb
wandb.login()

## Training

Stessi flag della checklist in `TODO.md`, con due differenze rispetto al primo run da 10 epoche:
- `--no_epochs 20`: nel primo run le metriche per-slice (Vol CC/KLDIV, il ramo temporale) stavano ancora migliorando in modo monotono all'ultima epoca, senza segni di appiattimento — c'è margine per continuare.
- `--model_val_path` punta a un file `_v2` diverso dal checkpoint del primo run (`multilevel_tempsal_ueyes.pt`), così quest'ultimo resta intatto come confronto.

Punti da ricordare (invariati):
- `--train_model 1` scongela la testa temporale di `pnas_vol`
- `--train_enc 0` **deve** essere esplicito — il default e' `1`, pensato per l'altro tipo di modello, e sbloccherebbe per sbaglio l'intero backbone PNAS
- `--model_path`/`--model_vol_path` puntano sempre al checkpoint originale (`multilevel_tempsal.pt`), non a quello fine-tuned: il warm-start riparte dal modello pre-UEyes, non incatena questo run al primo

Il codice clonato da GitHub include ora anche il fix al criterio di salvataggio del checkpoint: prima considerava solo CC/KLDIV sulla mappa aggregata (motivo per cui il primo run aveva smesso di salvare dopo l'epoca 7 pur avendo il ramo temporale ancora in miglioramento), ora include anche Vol CC/KLDIV.

Il checkpoint viene salvato **in locale** su Colab (piu' veloce, evita scritture lente/bloccanti sul mount di Drive durante il training); la cella successiva lo copia su Drive a fine training (o a intervalli, se la rilanci mentre il training e' ancora in corso).</cell id="IQNV-0zDgcXO">


In [ ]:
%cd /content/TempSal/src
!python train.py \
  --enc_model pnas_boosted_multi \
  --dataset_dir ../data_ueyes/ \
  --model_path ./checkpoints/multilevel_tempsal.pt \
  --model_vol_path ./checkpoints/multilevel_tempsal.pt \
  --train_model 1 \
  --train_enc 0 \
  --lr 1e-5 \
  --batch_size 32 \
  --no_epochs 20 \
  --model_val_path ./checkpoints/multilevel_tempsal_ueyes_v2.pt

In [ ]:
# Copia il checkpoint fine-tuned su Drive, cosi' sopravvive alla chiusura
# della sessione Colab. Puoi rieseguire questa cella anche a training ancora
# in corso, per avere un backup intermedio.
import shutil

src_ckpt = '/content/TempSal/src/checkpoints/multilevel_tempsal_ueyes_v2.pt'
dst_ckpt = f'{DRIVE_DIR}/multilevel_tempsal_ueyes_v2.pt'
shutil.copy(src_ckpt, dst_ckpt)
print('Copiato su Drive:', dst_ckpt)